# 未持贷客户 y_freq 概率预测

本 Notebook 使用分类预测任务中保存的最终模型、预处理器和元数据，对 2026-08-31 白名单直接预测。
采样方法由训练 Notebook 选择并写入模型元数据；可在配置区指定期望方法，避免误用其他模型。


In [ ]:
from pathlib import Path

# ===== 可修改配置 =====
PREDICT_FILE = Path(r"..\1-数据提取及清洗-0728修改版\whitelist_customer_extract_20260831.csv")
OUTPUT_FILE = Path("whitelist_customer_extract_20260831_y_freq_probability.csv")
MODEL_ROOT = Path(r"..\2-分类预测任务-0728修改版")
TARGET = "y_freq"
SNAPSHOT_DATE = "2026-08-31"

# auto：使用保存模型元数据中的采样方法；也可以填写 easy_ensemble、smote 等进行校验。
SAMPLING_METHOD = "auto"
CSV_ENCODING = "utf-8-sig"


In [ ]:
import sys
import json
import os
import joblib
import numpy as np
import pandas as pd

module_dir = MODEL_ROOT.resolve()
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
from load_consumer_loan_data import (
    read_data, rename_kechuang_cols, cast_cat_cols,
    clean_data_potential, drop_post_label_cols,
    PotentialFeaturePreprocessor,
)

if not PREDICT_FILE.exists():
    raise FileNotFoundError("未找到白名单文件：{0}".format(PREDICT_FILE.resolve()))
if not MODEL_ROOT.exists():
    raise FileNotFoundError("未找到分类预测目录：{0}".format(MODEL_ROOT.resolve()))

print("数据清洗模块：", module_dir / "load_consumer_loan_data.py")
raw = read_data(str(PREDICT_FILE), csv_encoding=CSV_ENCODING)
raw = rename_kechuang_cols(raw)
raw = cast_cat_cols(raw)
if "cst_id" not in raw.columns:
    raise KeyError("白名单数据缺少 cst_id")
raw["cst_id"] = raw["cst_id"].astype("string").str.strip()
if raw["cst_id"].isna().any() or raw["cst_id"].eq("").any():
    raise ValueError("白名单存在空客户编号")

# 与训练数据使用同一个清洗口径：brth_dt 按快照日生成 age，删除 ID/日期/泄露字段。
predict_clean = clean_data_potential(raw, snapshot_date=SNAPSHOT_DATE, target=TARGET)
predict_clean = drop_post_label_cols(predict_clean, target=TARGET)
predict_clean = predict_clean.drop_duplicates(subset=["cst_id"], keep="first").copy()
print("可预测客户数：{0:,}".format(len(predict_clean)))


In [ ]:
config_path = MODEL_ROOT / "selected_model_config_{0}.json".format(TARGET)
metadata_path = MODEL_ROOT / "full_model_metadata_{0}.json".format(TARGET)
preprocessor_path = MODEL_ROOT / "model_preprocessor_{0}.pkl".format(TARGET)
artifact_root = MODEL_ROOT / "full_model_{0}".format(TARGET)

for path in (config_path, metadata_path, preprocessor_path):
    if not path.exists():
        raise FileNotFoundError("缺少训练产物：{0}".format(path.resolve()))

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)
selected_method = metadata.get("selected_method_key", "baseline")
if SAMPLING_METHOD != "auto":
    requested = str(SAMPLING_METHOD).lower().replace("-", "_").replace(" ", "_")
    aliases = {"ensemble": "easy_ensemble", "easyensemble": "easy_ensemble"}
    requested = aliases.get(requested, requested)
    if requested != selected_method:
        raise ValueError(
            "配置的 SAMPLING_METHOD={0} 与保存模型方法={1} 不一致；"
            "请改为 auto，或先用该方法重新训练并保存模型。".format(requested, selected_method)
        )

preprocessor = joblib.load(preprocessor_path)
X_predict = preprocessor.transform(predict_clean)
feature_names = metadata.get("feature_names")
if feature_names is not None:
    missing = [c for c in feature_names if c not in X_predict.columns]
    if missing:
        raise KeyError("预测数据缺少训练模型字段：{0}".format(", ".join(missing)))
    X_predict = X_predict[feature_names]

model_count = int(metadata.get("model_count", 1))
models = []
import lightgbm as lgb
for i in range(model_count):
    model_path = artifact_root / "model_{0}.txt".format(i + 1)
    if not model_path.exists():
        raise FileNotFoundError("缺少模型成员：{0}".format(model_path.resolve()))
    models.append(lgb.Booster(model_file=str(model_path)))

rounds = metadata.get("selected_rounds")
probabilities = []
for model in models:
    kwargs = {}
    if rounds is not None:
        kwargs["num_iteration"] = int(rounds)
    probabilities.append(model.predict(X_predict, **kwargs))
pred_probability = np.mean(np.vstack(probabilities), axis=0)
if not np.isfinite(pred_probability).all():
    raise ValueError("预测概率出现 NaN 或无穷值")

result = pd.DataFrame({
    "cst_id": predict_clean["cst_id"].astype(str).to_numpy(),
    "pred_probability": pred_probability,
})
result = result.sort_values("pred_probability", ascending=False).reset_index(drop=True)
result["group"] = "id2"
top_n = max(1, int(np.ceil(len(result) * 0.10))) if len(result) else 0
if top_n:
    result.loc[:top_n - 1, "group"] = "id1"
result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("采样方法：", selected_method)
print("模型成员数：", len(models))
print("预测结果：", OUTPUT_FILE.resolve())
display(result.head(10))
